In [0]:
import json, random, time, uuid
from datetime import datetime

CAT, SCH = "capstone_five", "project"
OUT = "/Volumes/capstone_five/project/landing/vitals/"

# Pick 30 real patients from dim_patients
ids = [r[0] for r in spark.table(f"{CAT}.{SCH}.dim_patients").select("Id").limit(30).collect()]
print("Patients selected:", len(ids))

ROUNDS = 36          # 36 rounds x 5 sec = 3 minutes of data
for i in range(ROUNDS):
    rows = []
    for p in ids:
        hr = round(random.gauss(80, 10))              # normal heart rate
        if random.random() < 0.05:
            hr = random.randint(115, 160)              # ~5% tachycardia
        rows.append({
            "patient_id": p,
            "ts": datetime.utcnow().isoformat(),
            "heart_rate": hr,
            "glucose": round(random.gauss(110, 25)),
            "spo2": round(random.gauss(97, 1.5)),
        })
    dbutils.fs.put(f"{OUT}{uuid.uuid4()}.json", "\n".join(json.dumps(r) for r in rows))
    print(f"Batch {i+1}/{ROUNDS} written ({len(rows)} readings)")
    time.sleep(5)